In [30]:
import torch

In [31]:
D_in, H, D_out = 11, 100, 6

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)


In [32]:
outputs = model(torch.randn(600, 11))
outputs.shape

torch.Size([600, 6])

In [33]:
print(outputs[0][:])

tensor([ 0.7661,  0.6387, -0.1849, -0.1924, -0.0564, -0.0604],
       grad_fn=<SliceBackward0>)


In [34]:
model

Sequential(
  (0): Linear(in_features=11, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=6, bias=True)
)

In [35]:
model.to("cpu")

Sequential(
  (0): Linear(in_features=11, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=6, bias=True)
)

In [36]:
from sklearn.datasets import fetch_openml

# descarga datos

# Search for the red wine quality dataset on OpenML
# You can find dataset IDs on the OpenML website
# Let's assume the dataset ID for red wine quality is 186
# If this ID is incorrect, you might need to search OpenML for the correct one.
try:
    wine_quality = fetch_openml(name='wine-quality-red', version=1)
    X, Y = wine_quality["data"], wine_quality["target"]
    print("Dataset loaded successfully from OpenML.")
except Exception as e:
    print(f"Error loading dataset from OpenML: {e}")
    print("Please check the dataset name and version on OpenML.")
    # As a fallback, you might want to keep the previous loading method or provide instructions
    # to the user on how to find the correct dataset on OpenML.
    # For now, we'll just print the error and keep the original code as a comment.
    # mnist = fetch_openml('mnist_784', version=1)
    # X, Y = mnist["data"], mnist["target"]


X.shape, Y.shape

Dataset loaded successfully from OpenML.


((1599, 11), (1599,))

In [44]:
# normalización y split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import torch

# descarga datos
try:
    wine_quality = fetch_openml(name='wine-quality-red', version=1)
    X, Y = wine_quality["data"], wine_quality["target"]
    print("Dataset loaded successfully from OpenML.")
except Exception as e:
    print(f"Error loading dataset from OpenML: {e}")
    print("Please check the dataset name and version on OpenML.")

# Convert to numpy arrays for easier handling
x_2 = np.array(X)
y_2 = np.array(Y)

# Normalization and split
# The red wine dataset is much smaller, so we'll use a 80/20 split
X_train, X_test, y_train, y_test = train_test_split(x_2, y_2, test_size=0.2, random_state=42)

# The red wine dataset values are not pixel values (0-255), so we don't divide by 255.
# We will convert to float32 and keep the original values.
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.int32) # Ensure target is int32
y_test = y_test.astype(np.int32) # Ensure target is int32

# Adjust target labels to be 0-indexed (subtract 3)
y_train = y_train - 3
y_test = y_test - 3

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# convertimos datos a tensores
X_t = torch.from_numpy(X_train_scaled).float().cpu()
Y_t = torch.from_numpy(y_train).long().cpu()

print("X_train dtype:", X_train.dtype)
print("y_train dtype:", y_train.dtype)
print("X_train first 5 rows:\n", X_train[:5])
print("y_train first 5 values:\n", y_train[:5])
print("X_train_scaled first 5 rows:\n", X_train_scaled[:5])
print("X_t shape:", X_t.shape)
print("Y_t shape:", Y_t.shape)

# X_train, X_test, y_train, y_test = X[:60000] / 255., X[60000:] / 255., Y[:60000].astype(np.float32), Y[60000:].astype(np.float32)

Dataset loaded successfully from OpenML.
X_train dtype: float32
y_train dtype: int32
X_train first 5 rows:
 [[8.7000e+00 6.9000e-01 3.1000e-01 3.0000e+00 8.6000e-02 2.3000e+01
  8.1000e+01 1.0002e+00 3.4800e+00 7.4000e-01 1.1600e+01]
 [6.1000e+00 2.1000e-01 4.0000e-01 1.4000e+00 6.6000e-02 4.0500e+01
  1.6500e+02 9.9120e-01 3.2500e+00 5.9000e-01 1.1900e+01]
 [1.0900e+01 3.9000e-01 4.7000e-01 1.8000e+00 1.1800e-01 6.0000e+00
  1.4000e+01 9.9820e-01 3.3000e+00 7.5000e-01 9.8000e+00]
 [8.8000e+00 6.8500e-01 2.6000e-01 1.6000e+00 8.8000e-02 1.6000e+01
  2.3000e+01 9.9694e-01 3.3200e+00 4.7000e-01 9.4000e+00]
 [8.4000e+00 1.0350e+00 1.5000e-01 6.0000e+00 7.3000e-02 1.1000e+01
  5.4000e+01 9.9900e-01 3.3700e+00 4.9000e-01 9.9000e+00]]
y_train first 5 values:
 [3 3 3 2 2]
X_train_scaled first 5 rows:
 [[ 0.21833153  0.889712    0.19209224  0.30972564 -0.04964203  0.6910069
   1.0429336   1.8467128   1.0935      0.45822287  1.1231775 ]
 [-1.2901663  -1.7887825   0.6527534  -0.8050796  -0.45521

In [45]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def cross_entropy(output, target):
    logits = output[torch.arange(len(output)), target]
    loss = - logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    loss = loss.mean()
    return loss

In [46]:
#X_train

In [47]:
torch.cpu.is_available()

True

In [48]:
print(X)

      fixed_acidity  volatile_acidity  citric_acid  residual_sugar  chlorides  \
0               7.4             0.700         0.00             1.9      0.076   
1               7.8             0.880         0.00             2.6      0.098   
2               7.8             0.760         0.04             2.3      0.092   
3              11.2             0.280         0.56             1.9      0.075   
4               7.4             0.700         0.00             1.9      0.076   
...             ...               ...          ...             ...        ...   
1594            6.2             0.600         0.08             2.0      0.090   
1595            5.9             0.550         0.10             2.2      0.062   
1596            6.3             0.510         0.13             2.3      0.076   
1597            5.9             0.645         0.12             2.0      0.075   
1598            6.0             0.310         0.47             3.6      0.067   

      free_sulfur_dioxide  

In [49]:
# convertimos datos a tensores

X_t = torch.from_numpy(X_train_scaled).float().cpu()
Y_t = torch.from_numpy(y_train).long().cpu()

# Define criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

# bucle entrenamiento
epochs = 350
lr = 0.8
log_each = 10
l = []

model.to("cpu") # Move the model to CPU
model.train() # Set the model to training mode

for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t) # Use the built-in criterion
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad() # Use optimizer's zero_grad

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step() # Use optimizer's step

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 10/350 Loss 1.13611
Epoch 20/350 Loss 1.06864
Epoch 30/350 Loss 1.03503
Epoch 40/350 Loss 1.00990
Epoch 50/350 Loss 0.99191
Epoch 60/350 Loss 0.97810
Epoch 70/350 Loss 0.96682
Epoch 80/350 Loss 0.95725
Epoch 90/350 Loss 0.94890
Epoch 100/350 Loss 0.94146
Epoch 110/350 Loss 0.93481
Epoch 120/350 Loss 0.92879
Epoch 130/350 Loss 0.92328
Epoch 140/350 Loss 0.91816
Epoch 150/350 Loss 0.91341
Epoch 160/350 Loss 0.90896
Epoch 170/350 Loss 0.90478
Epoch 180/350 Loss 0.90086
Epoch 190/350 Loss 0.89710
Epoch 200/350 Loss 0.89351
Epoch 210/350 Loss 0.89007
Epoch 220/350 Loss 0.88674
Epoch 230/350 Loss 0.88359
Epoch 240/350 Loss 0.88054
Epoch 250/350 Loss 0.87756
Epoch 260/350 Loss 0.87468
Epoch 270/350 Loss 0.87189
Epoch 280/350 Loss 0.86919
Epoch 290/350 Loss 0.86654
Epoch 300/350 Loss 0.86398
Epoch 310/350 Loss 0.86150
Epoch 320/350 Loss 0.85909
Epoch 330/350 Loss 0.85673
Epoch 340/350 Loss 0.85445
Epoch 350/350 Loss 0.85222


In [50]:
# normalización y split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import torch

# descarga datos
try:
    wine_quality = fetch_openml(name='wine-quality-red', version=1)
    X, Y = wine_quality["data"], wine_quality["target"]
    print("Dataset loaded successfully from OpenML.")
except Exception as e:
    print(f"Error loading dataset from OpenML: {e}")
    print("Please check the dataset name and version on OpenML.")

# Convert to numpy arrays for easier handling
x_2 = np.array(X)
y_2 = np.array(Y)

# Normalization and split
# The red wine dataset is much smaller, so we'll use a 80/20 split
X_train, X_test, y_train, y_test = train_test_split(x_2, y_2, test_size=0.2, random_state=42)

# The red wine dataset values are not pixel values (0-255), so we don't divide by 255.
# We will convert to float32 and keep the original values.
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.int32) # Ensure target is int32
y_test = y_test.astype(np.int32) # Ensure target is int32

# Adjust target labels to be 0-indexed (subtract 3)
y_train = y_train - 3
y_test = y_test - 3

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# convertimos datos a tensores
X_t = torch.from_numpy(X_train_scaled).float().cpu()
Y_t = torch.from_numpy(y_train).long().cpu()

print("X_train dtype:", X_train.dtype)
print("y_train dtype:", y_train.dtype)
print("X_train first 5 rows:\n", X_train[:5])
print("y_train first 5 values:\n", y_train[:5])
print("X_train_scaled first 5 rows:\n", X_train_scaled[:5])
print("X_t shape:", X_t.shape)
print("Y_t shape:", Y_t.shape)

# X_train, X_test, y_train, y_test = X[:60000] / 255., X[60000:] / 255., Y[:60000].astype(np.float32), Y[60000:].astype(np.float32)

Dataset loaded successfully from OpenML.
X_train dtype: float32
y_train dtype: int32
X_train first 5 rows:
 [[8.7000e+00 6.9000e-01 3.1000e-01 3.0000e+00 8.6000e-02 2.3000e+01
  8.1000e+01 1.0002e+00 3.4800e+00 7.4000e-01 1.1600e+01]
 [6.1000e+00 2.1000e-01 4.0000e-01 1.4000e+00 6.6000e-02 4.0500e+01
  1.6500e+02 9.9120e-01 3.2500e+00 5.9000e-01 1.1900e+01]
 [1.0900e+01 3.9000e-01 4.7000e-01 1.8000e+00 1.1800e-01 6.0000e+00
  1.4000e+01 9.9820e-01 3.3000e+00 7.5000e-01 9.8000e+00]
 [8.8000e+00 6.8500e-01 2.6000e-01 1.6000e+00 8.8000e-02 1.6000e+01
  2.3000e+01 9.9694e-01 3.3200e+00 4.7000e-01 9.4000e+00]
 [8.4000e+00 1.0350e+00 1.5000e-01 6.0000e+00 7.3000e-02 1.1000e+01
  5.4000e+01 9.9900e-01 3.3700e+00 4.9000e-01 9.9000e+00]]
y_train first 5 values:
 [3 3 3 2 2]
X_train_scaled first 5 rows:
 [[ 0.21833153  0.889712    0.19209224  0.30972564 -0.04964203  0.6910069
   1.0429336   1.8467128   1.0935      0.45822287  1.1231775 ]
 [-1.2901663  -1.7887825   0.6527534  -0.8050796  -0.45521

In [51]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled first 5 rows:\n", X_train_scaled[:5])

X_train_scaled first 5 rows:
 [[ 0.21833153  0.889712    0.19209224  0.30972564 -0.04964203  0.6910069
   1.0429336   1.8467128   1.0935      0.45822287  1.1231775 ]
 [-1.2901663  -1.7887825   0.6527534  -0.8050796  -0.45521364  2.388473
   3.5938704  -3.0045054  -0.40043876 -0.40119714  1.4082713 ]
 [ 1.4947526  -0.7843472   1.0110453  -0.52637833  0.5992724  -0.9579601
  -0.991742    0.7686535  -0.07566977  0.5155175  -0.5873896 ]
 [ 0.2763509   0.86181104 -0.0638307  -0.665729   -0.00908494  0.01202048
  -0.7184274   0.0894971   0.05423781 -1.0887328  -0.9675162 ]
 [ 0.04427397  2.81488    -0.6268609   2.3999856  -0.3132636  -0.47296986
   0.2229897   1.1998773   0.37900677 -0.9741434  -0.49235862]]


In [52]:
import numpy as np

unique_values, counts = np.unique(y_train, return_counts=True)
print("Unique values in y_train:", unique_values)
print("Counts of unique values in y_train:", counts)

Unique values in y_train: [0 1 2 3 4 5]
Counts of unique values in y_train: [  9  43 551 506 157  13]


In [53]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
accuracy_score(y_test, y_pred.cpu().numpy())

0.409375

In [54]:
criterion = torch.nn.CrossEntropyLoss()

In [55]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

In [56]:
D_in, H, D_out = 11, 100, 6
model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cpu")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 1500
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/1500 Loss 1.18926
Epoch 20/1500 Loss 1.11132
Epoch 30/1500 Loss 1.06141
Epoch 40/1500 Loss 1.03020
Epoch 50/1500 Loss 1.00826
Epoch 60/1500 Loss 0.99146
Epoch 70/1500 Loss 0.97778
Epoch 80/1500 Loss 0.96636
Epoch 90/1500 Loss 0.95670
Epoch 100/1500 Loss 0.94834
Epoch 110/1500 Loss 0.94094
Epoch 120/1500 Loss 0.93425
Epoch 130/1500 Loss 0.92814
Epoch 140/1500 Loss 0.92252
Epoch 150/1500 Loss 0.91734
Epoch 160/1500 Loss 0.91254
Epoch 170/1500 Loss 0.90803
Epoch 180/1500 Loss 0.90378
Epoch 190/1500 Loss 0.89978
Epoch 200/1500 Loss 0.89598
Epoch 210/1500 Loss 0.89237
Epoch 220/1500 Loss 0.88888
Epoch 230/1500 Loss 0.88556
Epoch 240/1500 Loss 0.88240
Epoch 250/1500 Loss 0.87936
Epoch 260/1500 Loss 0.87646
Epoch 270/1500 Loss 0.87368
Epoch 280/1500 Loss 0.87098
Epoch 290/1500 Loss 0.86835
Epoch 300/1500 Loss 0.86581
Epoch 310/1500 Loss 0.86332
Epoch 320/1500 Loss 0.86090
Epoch 330/1500 Loss 0.85855
Epoch 340/1500 Loss 0.85626
Epoch 350/1500 Loss 0.85404
Epoch 360/1500 Loss 0.85186
E

0.3875

In [63]:
from sklearn.metrics import accuracy_score
import torch
import numpy as np

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def evaluate(x):
    model.eval()
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)

class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

model = ModelCustom2(11, 100, 6).to("cpu")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 1.16391
Epoch 20/100 Loss 1.06685
Epoch 30/100 Loss 1.02733
Epoch 40/100 Loss 1.00443
Epoch 50/100 Loss 0.98889
Epoch 60/100 Loss 0.97736
Epoch 70/100 Loss 0.96829
Epoch 80/100 Loss 0.96087
Epoch 90/100 Loss 0.95461
Epoch 100/100 Loss 0.94920


0.3875

In [64]:
model = ModeloPersonalizado(11, 100, 6)
# Codigo para saber si el modelo esta votando los datos en las cantidades correctas
x_prueba = torch.randn(500, 11)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[-0.7312,  1.8533,  0.4050,  ..., -1.1430,  0.0075,  0.2564],
        [ 0.3962,  0.6068,  0.2948,  ..., -0.2341, -1.1112,  0.1097],
        [ 0.0542, -1.0152,  0.0681,  ...,  1.8481, -1.4203,  0.2767],
        ...,
        [-1.0260,  0.7355,  0.4155,  ...,  0.9304,  0.6405,  1.5436],
        [ 0.2802,  0.3474,  0.6871,  ...,  1.3278,  0.7925, -0.6714],
        [ 1.3465,  1.3108,  0.1316,  ...,  0.8950,  1.7795, -0.4998]])


torch.Size([500, 6])

In [65]:
model.to("cpu")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 1.16814
Epoch 20/100 Loss 1.08806
Epoch 30/100 Loss 1.04575
Epoch 40/100 Loss 1.01931
Epoch 50/100 Loss 1.00027
Epoch 60/100 Loss 0.98560
Epoch 70/100 Loss 0.97369
Epoch 80/100 Loss 0.96358
Epoch 90/100 Loss 0.95477
Epoch 100/100 Loss 0.94705


0.409375

In [66]:
model.to("cpu")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 0.87091
Epoch 20/100 Loss 0.86773
Epoch 30/100 Loss 0.86467
Epoch 40/100 Loss 0.86184
Epoch 50/100 Loss 0.85912
Epoch 60/100 Loss 0.85648
Epoch 70/100 Loss 0.85397
Epoch 80/100 Loss 0.85156
Epoch 90/100 Loss 0.84925
Epoch 100/100 Loss 0.84703


0.4125

In [67]:
model = ModelCustom2(11, 100, 6).to("cpu")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 1.14965
Epoch 20/100 Loss 1.05676
Epoch 30/100 Loss 1.01949
Epoch 40/100 Loss 0.99812
Epoch 50/100 Loss 0.98371
Epoch 60/100 Loss 0.97308
Epoch 70/100 Loss 0.96475
Epoch 80/100 Loss 0.95795
Epoch 90/100 Loss 0.95221
Epoch 100/100 Loss 0.94726


0.39375

In [68]:
model

ModelCustom2(
  (fc1): Linear(in_features=11, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=6, bias=True)
)

In [69]:
model.fc1

Linear(in_features=11, out_features=100, bias=True)

In [70]:
model.fc1.weight

Parameter containing:
tensor([[-0.2068,  0.0473,  0.0246,  ..., -0.1610,  0.0781,  0.0097],
        [ 0.1506,  0.0310, -0.0234,  ..., -0.2778, -0.1287,  0.0530],
        [ 0.1798, -0.0941, -0.1058,  ...,  0.1989,  0.2540,  0.2553],
        ...,
        [ 0.1659,  0.1855, -0.2915,  ..., -0.1799, -0.0149, -0.0272],
        [ 0.2870, -0.2734, -0.2251,  ...,  0.1616,  0.2817,  0.0486],
        [ 0.1535,  0.1197, -0.2309,  ..., -0.2678,  0.0284, -0.0141]],
       requires_grad=True)

In [71]:
model.fc1.bias

Parameter containing:
tensor([-0.0706,  0.0398,  0.1223,  0.0506,  0.2519,  0.2256, -0.2136,  0.1724,
         0.0484,  0.1297, -0.2626, -0.2850, -0.0329, -0.1595, -0.2463, -0.2184,
         0.0144,  0.3221, -0.1680,  0.2452, -0.2721, -0.0008,  0.3132, -0.1122,
        -0.2444, -0.2502,  0.0491, -0.2425, -0.1543,  0.0722,  0.1090,  0.0431,
        -0.0233,  0.2395, -0.0350,  0.2329, -0.2322,  0.1980, -0.2289, -0.2097,
        -0.0991, -0.2404, -0.1351,  0.2310, -0.1828, -0.2185,  0.3194, -0.0550,
        -0.2166, -0.0273,  0.2664,  0.0932,  0.0494, -0.2858,  0.2720,  0.1634,
        -0.0300,  0.3559,  0.2874, -0.1599,  0.0284, -0.0974,  0.0852,  0.3055,
         0.2916,  0.2785, -0.0420, -0.0323,  0.1855,  0.2318,  0.2383, -0.1299,
         0.1646,  0.3478, -0.2617,  0.0676,  0.1123,  0.2158, -0.1814,  0.0061,
        -0.2133,  0.2926,  0.2960,  0.0219,  0.2832,  0.2608,  0.0550, -0.0464,
         0.0280,  0.0169, -0.1727,  0.0928,  0.1257,  0.2797,  0.0523, -0.1000,
        -0.2880,  

In [72]:
model.fc2 = torch.nn.Linear(100, 1)

model

ModelCustom2(
  (fc1): Linear(in_features=11, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

In [73]:
# obtener una lista con las capas de una red

list(model.children())

[Linear(in_features=11, out_features=100, bias=True),
 ReLU(),
 Linear(in_features=100, out_features=1, bias=True)]

In [74]:
# crear nueva red a partir de la lista (excluyendo las útlimas dos capa)

new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

Sequential(
  (0): Linear(in_features=11, out_features=100, bias=True)
)

In [75]:
# crear nueva red a partir de la lista (excluyendo las útlima capa)

new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

ModuleList(
  (0): Linear(in_features=11, out_features=100, bias=True)
  (1): ReLU()
)